# Isi tabel setelah diolah (`data/processed/`)

Notebook ini cuma untuk **melihat isi** 4 dataset hasil `dataprep.build`, yaitu data yang dimuat ke Postgres dan dipakai chatbot.
Tidak mengubah data apa pun. Jalankan dari atas ke bawah (Run All).

| Dataset | Dari | Dipakai untuk |
|---|---|---|
| `darglobal_projects` | halaman proyek DarGlobal | tabel proyek (filter SQL) |
| `darglobal_chunks` | semua halaman DarGlobal (blog, press, umum, proyek) | pencarian teks (RAG) |
| `wasalt_listings` | listing Wasalt yang lolos pembersihan | tabel listing (filter SQL) |
| `wasalt_rejected` | listing Wasalt yang dibuang | catatan saja |

In [ ]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 100)

PROCESSED = Path("..") / "data" / "processed" if Path("../data/processed").exists() else Path("data/processed")


def load(name: str) -> pd.DataFrame:
    with (PROCESSED / f"{name}.jsonl").open(encoding="utf-8") as f:
        return pd.DataFrame([json.loads(line) for line in f if line.strip()])


projects = load("darglobal_projects")
chunks = load("darglobal_chunks")
listings = load("wasalt_listings")
rejected = load("wasalt_rejected")

pd.DataFrame(
    {"baris": [len(projects), len(chunks), len(listings), len(rejected)],
     "kolom": [projects.shape[1], chunks.shape[1], listings.shape[1], rejected.shape[1]]},
    index=["darglobal_projects", "darglobal_chunks", "wasalt_listings", "wasalt_rejected"],
)

## 1. `darglobal_projects`: 1 baris = 1 proyek DarGlobal

In [ ]:
# Semua 40 proyek, kolom utama saja
projects[["slug", "title", "country", "location", "property_type", "status", "starting_price", "completion"]]

In [ ]:
# Semua kolom + berapa baris yang terisi (sisanya kosong/None)
pd.DataFrame({"terisi": projects.notna().sum(), "dari": len(projects)})

In [ ]:
# Satu proyek utuh, semua kolom (ganti angka 0 untuk proyek lain)
projects.iloc[0].to_frame("isi")

In [ ]:
display(projects["country"].value_counts(dropna=False).rename("jumlah proyek per negara"))
display(projects["status"].value_counts(dropna=False).rename("jumlah proyek per status"))

## 2. `darglobal_chunks`: 1 baris = 1 potongan teks (bahan RAG)

In [ ]:
display(chunks["kind"].value_counts().rename("jumlah chunk per jenis halaman"))
chunks[["chunk_id", "kind", "title", "section", "url"]].head(10)

In [ ]:
# Isi teks lengkap 1 chunk (ganti angka untuk chunk lain)
row = chunks.iloc[0]
print(row["url"], "|", row["kind"], "|", row["section"], "\n")
print(row["text"])

In [ ]:
# Panjang teks per chunk (jumlah karakter)
chunks["text"].str.len().describe()

## 3. `wasalt_listings`: 1 baris = 1 listing properti Wasalt

In [ ]:
listings.head(10)

In [ ]:
display(listings["purpose"].value_counts().rename("jual vs sewa"))
display(listings["city"].value_counts().head(10).rename("10 kota terbanyak"))
display(listings["property_type"].value_counts().head(10).rename("10 tipe properti terbanyak"))

In [ ]:
# Harga (SAR): jual = harga total, sewa = harga sewa per tahun
sale = listings[listings["purpose"] == "sale"]["price_sar"].describe()
rent = listings[listings["purpose"] == "rent"]["rent_yearly_sar"].describe()
pd.DataFrame({"jual (price_sar)": sale, "sewa per tahun (rent_yearly_sar)": rent})

## 4. `wasalt_rejected`: listing yang dibuang saat pembersihan

In [ ]:
display(rejected["reason"].value_counts().rename("alasan dibuang"))
rejected.head(10)

## 5. Coba cari sendiri

Ubah kata/angka di bawah, lalu jalankan.

In [ ]:
# Proyek DarGlobal di negara tertentu
projects.loc[projects["country"] == "Oman", ["title", "property_type", "status", "location"]]

In [ ]:
# Chunk yang mengandung kata tertentu (misal "golf")
kata = "golf"
hit = chunks[chunks["text"].str.contains(kata, case=False, na=False)]
print(len(hit), "chunk mengandung", repr(kata))
hit[["kind", "title", "section", "url"]].head(10)

In [ ]:
# Listing Wasalt: jual di Riyadh, harga di bawah 1 juta SAR
listings.loc[
    (listings["purpose"] == "sale") & (listings["city"] == "Riyadh") & (listings["price_sar"] < 1_000_000),
    ["title", "property_type", "district", "price_sar", "area_sqm", "bedrooms"],
].sort_values("price_sar").head(10)